# Badge 4: Data Lake Workshop: Lesson 6: GeoSpatial Views

CHERRY_CREEK_TRAIL and DENVER_AREA_TRAILS.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE4_DLKW_ADMIN` (ACCOUNTADMIN), `BADGE4_DLKW_ROLE` (SYSADMIN) and `BADGE4_DLKW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.BADGE4_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE4_DLKW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE4_DLKW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE4_DLKW_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
use schema MELS_SMOOTHIE_CHALLENGE_DB.TRAILS;
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: 🥋 Querying the Parquet File

> **Course:** Write a more sophisticated query to parse the data into columns (DLKW_086).

In [ ]:
%%sql -r parquet_columns
-- From screenshot DLKW_086, blur filled in
select
  $1:sequence_1 as sequence_1,
  $1:trail_name::varchar as trail_name,
  $1:latitude as latitude,
  $1:longitude as longitude,
  $1:sequence_2 as sequence_2,
  $1:elevation as elevation
from @trails_parquet
(file_format => ff_parquet)
order by sequence_1;

## Page: 🎯 Create a View with Nice Looking GeoSpatial Columns

In [ ]:
%%sql -r nicely_formatted
--Nicely formatted trail data
select 
 $1:sequence_1 as point_id,
 $1:trail_name::varchar as trail_name,
 $1:latitude::number(11,8) as lng, --remember we did a gut check on this data
 $1:longitude::number(11,8) as lat
from @trails_parquet
(file_format => ff_parquet)
order by point_id;

> ⚠️ **Audit note:** The course's select deliberately maps `$1:latitude` to `lng` and `$1:longitude` to `lat` (the file has them swapped; the code comment says "we did a gut check on this data").

> **Course:** 🎯 Create a View Called CHERRY_CREEK_TRAIL: wrap the select in CREATE VIEW, in Mel's database and TRAILS schema, owned by SYSADMIN.

In [ ]:
%%sql -r create_cherry_creek_trail
-- Derived solution
create view cherry_creek_trail as
--Nicely formatted trail data
select 
 $1:sequence_1 as point_id,
 $1:trail_name::varchar as trail_name,
 $1:latitude::number(11,8) as lng, --remember we did a gut check on this data
 $1:longitude::number(11,8) as lat
from @trails_parquet
(file_format => ff_parquet)
order by point_id;

## Page: 🥋 Replace It With a Better View

In [ ]:
%%sql -r coord_pairs
--Using concatenate to prepare the data for plotting on a map
select top 100 
 lng||' '||lat as coord_pair
,'POINT('||coord_pair||')' as trail_point
from cherry_creek_trail;

In [ ]:
%%sql -r replace_cherry_creek_trail
--To add a column, we have to replace the entire view
--changes to the original are shown in red
create or replace view cherry_creek_trail as
select 
 $1:sequence_1 as point_id,
 $1:trail_name::varchar as trail_name,
 $1:latitude::number(11,8) as lng,
 $1:longitude::number(11,8) as lat,
 lng||' '||lat as coord_pair
from @trails_parquet
(file_format => ff_parquet)
order by point_id;

## Page: 🥋 Can We Generate a LINESTRING( )?

In [ ]:
%%sql -r linestring_10
select 
'LINESTRING('||
listagg(coord_pair, ',') 
within group (order by point_id)
||')' as my_linestring
from cherry_creek_trail
where point_id <= 10
group by trail_name;

## Page: 🎯 Generate the Full Trail in a single LINESTRING

In [ ]:
%%sql -r linestring_full
-- Derived solution: change the limit from 10 to 2450 (WKT Playground errors on all 3,500 points)
select 
'LINESTRING('||
listagg(coord_pair, ',') 
within group (order by point_id)
||')' as my_linestring
from cherry_creek_trail
where point_id <= 2450
group by trail_name;

## Page: 🥋 Explore the geoJSON Files

In [ ]:
%%sql -r explore_geojson
-- DLKW_095: a select on the geoJSON stage with the JSON file format
select * from @trails_geojson (file_format => ff_json);

In [ ]:
%%sql -r normalize_geojson
select
$1:features[0]:properties:Name::string as feature_name
,$1:features[0]:geometry:coordinates::string as feature_coordinates
,$1:features[0]:geometry::string as geometry
,$1:features[0]:properties::string as feature_properties
,$1:crs:properties:name::string as specs
,$1 as whole_object
from @trails_geojson (file_format => ff_json);

## Page: 🎯 Create a View for the geoJSON files

> **Course:** 🎯 Create a View Called DENVER_AREA_TRAILS: wrap the previous select in CREATE VIEW, in Mel's database and TRAILS schema, owned by SYSADMIN.

In [ ]:
%%sql -r create_denver_area_trails
-- Derived solution
create view DENVER_AREA_TRAILS as
select
$1:features[0]:properties:Name::string as feature_name
,$1:features[0]:geometry:coordinates::string as feature_coordinates
,$1:features[0]:geometry::string as geometry
,$1:features[0]:properties::string as feature_properties
,$1:crs:properties:name::string as specs
,$1 as whole_object
from @trails_geojson (file_format => ff_json);

## Page: 🤖 DORA DLKW06

### 🤖 DORA check DLKW06 goes here
The course has the learner run **DLKW06** (CHERRY_CREEK_TRAIL and DENVER_AREA_TRAILS views exist) at this point.
It is in `dora/dora_tests.ipynb`, section **DLKW06**, as a local check plus the exact DORA code.